# 01 · Bronze: conferência dos arquivos brutos

Confere existência, MD5, bytes, registros e colunas contra `data/bronze/_catalogo_bronze.csv`.

CSV: detecta vírgula, ponto e vírgula ou tabulação; leitura UTF-8 ou cp1252. GeoJSON: conta feições e propriedades (sem geometria). ZIP-Shapefile: confere CRC, cabeçalhos, registros DBF/SHX e atributos DBF, sem validar a topologia. ZIP com um CSV: confere CRC e lê os registros internos sem extrair. Entradas restritas do catálogo exigem os arquivos locais em _restrito/, que não vão para o Git. Esta conferência de recebimento não valida geometrias ou cobertura territorial.

**Regra do Bronze:** o arquivo bruto nunca é editado. Arquivo novo do portal entra como lote novo, com nova linha no catálogo.


In [1]:
import sys
from pathlib import Path

# Acha a raiz do repositório (a pasta que tem src/) e deixa o pacote wastestack importável
RAIZ = Path.cwd()
while not (RAIZ / "src" / "wastestack").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

In [2]:
import pandas as pd
from wastestack.config import RAIZ, BRONZE
from wastestack.bronze import inspecionar_arquivo

catalogo = pd.read_csv(BRONZE / "_catalogo_bronze.csv", dtype=str)
catalogo[["dataset", "arquivo", "recebido_em", "linhas_de_dados"]]


                   dataset                                                                                  arquivo recebido_em linhas_de_dados
0                      156                                                             data/bronze/156/156_2024.csv  2026-11-10          108082
1                      156                                                             data/bronze/156/156_2025.csv  2026-11-10           86966
2                      156                                                             data/bronze/156/156_2026.csv  2026-11-10           61225
3              area_urbana                 data/bronze/area_urbana/2026-11-11/trechos-de-logradouros-por-bairro.csv  2026-11-11           11921
4              area_urbana                             data/bronze/area_urbana/2026-11-11/bairros-do-recife.geojson  2026-11-11              94
5              area_urbana                        data/bronze/area_urbana/2026-11-11/trechos-de-logradouros.geojson  2026-11-11         

In [3]:
resultado = []
for _, linha in catalogo.iterrows():
    caminho = RAIZ / linha["arquivo"]
    existe = caminho.exists()
    info = inspecionar_arquivo(caminho) if existe else {}
    resultado.append({
        "arquivo": linha["arquivo"],
        "existe": existe,
        "hash_confere": info.get("md5") == linha["md5"],
        "bytes_conferem": str(info.get("bytes")) == linha["bytes"],
        "linhas_conferem": str(info.get("linhas_de_dados")) == linha["linhas_de_dados"],
        "colunas_conferem": str(info.get("colunas")) == linha["colunas"],
        "linhas": info.get("linhas_de_dados"),
    })

conferencia = pd.DataFrame(resultado)
conferencia


                                                                                    arquivo  existe  hash_confere  bytes_conferem  linhas_conferem  colunas_conferem  linhas
0                                                              data/bronze/156/156_2024.csv    True          True            True             True              True  108082
1                                                              data/bronze/156/156_2025.csv    True          True            True             True              True   86966
2                                                              data/bronze/156/156_2026.csv    True          True            True             True              True   61225
3                  data/bronze/area_urbana/2026-11-11/trechos-de-logradouros-por-bairro.csv    True          True            True             True              True   11921
4                              data/bronze/area_urbana/2026-11-11/bairros-do-recife.geojson    True          True            True      

In [4]:
assert conferencia["existe"].all(), "Algum arquivo do catálogo não está na pasta"
assert conferencia["hash_confere"].all(), "Algum arquivo mudou desde que entrou no catálogo"
assert conferencia["bytes_conferem"].all(), "O tamanho não bate com o catálogo"
assert conferencia["linhas_conferem"].all(), "O número de registros não bate com o catálogo"
assert conferencia["colunas_conferem"].all(), "O número de colunas não bate com o catálogo"
print("Bronze OK")


Bronze OK
